# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [2]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

Fichero: c:\MLOps\icai-muiaap-operacion-de-modelos\semana3\assets\03-wine-quality\inference_samples.csv
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Filas de ejemplo: 5


## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | No disponer del modelo ejecutable|
| Versión del esquema | **manifest.json** | Leer un manifiesto con formato incompatible |
| Orden y nombres de features |**model.joblib**| Introducir características incorrectas o desordenadas |
| Versión del preprocesado | **manifest.json** | Utilizar un preprocesado incompatible |
| Etiquetas de salida permitidas | **manifest.json** | Aceptar predicciones no permitidas |
| Versión del modelo | **model.joblib** | Utilizar un modelo de versión incorrecta |
| Tipo de estimador | **manifest.json** | Cargar un estimador de tipo inesperado |

Regla extra del checkpoint 1: estimator_type declarado en el manifiesto debe coincidir con el tipo real del estimador.

Condiciones que bloqueen la carga o inferencia:

manifest.feature_names debe coincidir exactamente con las features y su orden de la Semana 3.

manifest.preprocessing_version debe ser compatible con la versión del preprocesado.

Las predicciones deben pertenecer a manifest.output_labels.

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "1.0",
  "model_version": "wine-quality-rf-demo-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": ["fixed_acidity",
                  "volatile_acidity",
                  "citric_acid",
                  "residual_sugar",
                  "chlorides",
                  "free_sulfur_dioxide",
                  "total_sulfur_dioxide",
                  "density",
                  "ph",
                  "sulphates",
                  "alcohol",],
  "output_labels": ["needs_review", "acceptable", "excellent"],
  "estimator_type": "RandomForestClassifier"
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**;
2. aparecen features desconocidas o en un orden distinto;
3. el modelo devuelve una etiqueta que no está en **output_labels**;
4. una fila del CSV no cumple el contrato de entrada.

In [6]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-demo-v1"

manifest_draft = {
    "schema_version": "1.0",
    "model_version": legacy_model_version,
    "preprocessing_version": "wine-red-features-v1",
    "feature_names": feature_names,
    "output_labels": ["low", "medium", "high"],
    "estimator_type": "RandomForestClassifier",
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.")

{
  "schema_version": "1.0",
  "model_version": "wine-quality-rf-demo-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "low",
    "medium",
    "high"
  ],
  "estimator_type": "RandomForestClassifier"
}

La estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.


## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | Contrato del artefacto | Feature order incompatible | No |
| Cambia `preprocessing_version` | Contrato del artefacto | Preprocessing version incompatible | No |
| El modelo devuelve `unknown` | Contrato de salida | Unknown output label | No |
| La petición incluye `ph = 99` | Contrato de entrada | ph out of range | No |
| Tipo de estimador diferente al declarado | Contrato del artefacto |	Estimator type incompatible | No |

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
schema_version
model_version
preprocessing_version
feature_names
output_labels
estimator_type
Ninguno tiene valores por defecto
- ¿Aceptarás campos desconocidos en el manifiesto?
No, siguiendo lo que hicimos en el contrato de entrada de la Semana 3, si recibimos un manifiesto que contiene una clave que no hemos definido, Pydantic genera un error.
- ¿Validarás el manifiesto antes de cargar el objeto serializado?
Si
- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
Solo después de validar todas las filas y predicciones.
- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
Un mensaje claro y específico que explique el motivo del error como los valores recibidos y los esperados.
- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?
Debe detenerse el procesamiento, informar del error y no escribir el archivo CSV de salida.

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.

El loader debe comprobar que el estimador dispone de los métodos predict y predict_proba antes de permitir la inferencia.